# PPE Detection: 4-model comparison on a Colab GPU

Cleans and re-splits the dataset (no leakage), trains **YOLOv5s, YOLOv8n, YOLOv8s and YOLO11n**,
then evaluates all four on the same test split with tables and figures.

**Before you start:** `Runtime > Change runtime type > T4 GPU`.

**Dataset:** put `css-data.zip` (the Kaggle download) in the root of your Google Drive (`My Drive/css-data.zip`).

Total time on a T4 is roughly 1.5-2.5 hours at 30 epochs. Keep this tab open.

## 1. GPU check and install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics
%cd /content

## 2. Project scripts (identical to the ones in the GitHub repo)

In [ ]:
%%writefile data_cleaning.py
"""Data cleaning, validation and analysis for a YOLO-format dataset.

    python data_cleaning.py --data dataset/data.yaml --out figures/data_raw
    python data_cleaning.py --data dataset_clean/data.yaml --out figures/data_clean
    (--fix also repairs labels in place)

Checks: corrupt images, images without labels, labels without images, malformed
label lines, out-of-range class IDs and coordinates, zero-size boxes, duplicate
boxes, and duplicate images across splits (data leakage).
Writes figures (class distribution, split sizes, box statistics, sample grid)
and cleaning_report.json to --out.
"""
import argparse
import hashlib
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import yaml
from PIL import Image, ImageDraw

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
SPLITS = {"train": "train", "val": "valid", "test": "test"}


def check_label_file(path, nc):
    """Return (valid_boxes, list of problems) for one label file."""
    boxes, problems, seen = [], [], set()
    for n, line in enumerate(path.read_text().splitlines(), 1):
        parts = line.split()
        if not parts:
            continue
        if len(parts) != 5:
            problems.append(f"line {n}: expected 5 values, got {len(parts)}")
            continue
        try:
            cls, x, y, w, h = int(parts[0]), *map(float, parts[1:])
        except ValueError:
            problems.append(f"line {n}: not numeric")
            continue
        if not 0 <= cls < nc:
            problems.append(f"line {n}: class {cls} out of range")
        elif not all(0 <= v <= 1 for v in (x, y, w, h)):
            problems.append(f"line {n}: coordinates outside [0,1]")
        elif w <= 0 or h <= 0:
            problems.append(f"line {n}: zero-size box")
        elif (cls, x, y, w, h) in seen:
            problems.append(f"line {n}: duplicate box")
        else:
            seen.add((cls, x, y, w, h))
            boxes.append((cls, x, y, w, h))
    return boxes, problems


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", required=True)
    p.add_argument("--fix", action="store_true", help="rewrite labels without bad lines, drop corrupt images")
    p.add_argument("--out", default="figures/data", help="folder for figures and cleaning_report.json")
    args = p.parse_args()

    cfg = yaml.safe_load(Path(args.data).read_text(encoding="utf-8"))
    root, names = Path(args.data).parent, cfg["names"]
    nc = len(names)
    figs = Path(args.out)
    figs.mkdir(parents=True, exist_ok=True)

    report = {"splits": {}, "issues": defaultdict(list), "fixed": args.fix}
    class_counts = {s: Counter() for s in SPLITS}
    boxes_per_img, box_areas, hashes = [], [], defaultdict(list)
    all_boxes = {}

    for split, folder in SPLITS.items():
        img_dir, lbl_dir = root / folder / "images", root / folder / "labels"
        imgs = {f.stem: f for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXT}
        lbls = {f.stem: f for f in lbl_dir.glob("*.txt")}

        for stem in sorted(set(lbls) - set(imgs)):
            report["issues"]["label_without_image"].append(f"{split}/{stem}")
        for stem, img in sorted(imgs.items()):
            try:
                with Image.open(img) as im:
                    im.verify()
            except Exception as e:
                report["issues"]["corrupt_image"].append(f"{split}/{img.name}: {e}")
                if args.fix:
                    img.unlink()
                continue
            hashes[hashlib.md5(img.read_bytes()).hexdigest()].append(f"{split}/{img.name}")

            if stem not in lbls:
                report["issues"]["image_without_label"].append(f"{split}/{img.name}")
                boxes = []
            else:
                boxes, problems = check_label_file(lbls[stem], nc)
                for pr in problems:
                    report["issues"]["bad_label_line"].append(f"{split}/{stem}.txt {pr}")
                if problems and args.fix:
                    lbls[stem].write_text("".join(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n"
                                                  for c, x, y, w, h in boxes))
                if not boxes:
                    report["issues"]["empty_label (background image)"].append(f"{split}/{img.name}")

            all_boxes[img] = boxes
            boxes_per_img.append(len(boxes))
            for c, _, _, w, h in boxes:
                class_counts[split][names[c]] += 1
                box_areas.append(w * h)
        report["splits"][split] = {"images": len(imgs), "instances": sum(class_counts[split].values())}

    dupes = [v for v in hashes.values() if len(v) > 1]
    cross = [v for v in dupes if len({x.split("/")[0] for x in v}) > 1]
    report["issues"]["duplicate_images"] = dupes
    report["issues"]["duplicates_across_splits (leakage)"] = cross

    total = sum(r["images"] for r in report["splits"].values())
    for r in report["splits"].values():
        r["percent"] = round(100 * r["images"] / total, 1)
    report["class_counts"] = {s: dict(c) for s, c in class_counts.items()}
    report["summary"] = {k: len(v) for k, v in report["issues"].items()}

    # ---- figures ----
    fig, ax = plt.subplots(figsize=(10, 4.5))
    width = 0.27
    for i, split in enumerate(SPLITS):
        ax.bar([j + (i - 1) * width for j in range(nc)], [class_counts[split][n] for n in names],
               width, label=split)
    ax.set_xticks(range(nc), names, rotation=35, ha="right")
    ax.set_ylabel("Instances")
    ax.set_title("Class distribution per split")
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(figs / "class_distribution.png", dpi=150)

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    axes[0].hist(boxes_per_img, bins=30, color="#4C72B0")
    axes[0].set(title="Boxes per image", xlabel="Boxes", ylabel="Images")
    axes[1].hist([a * 100 for a in box_areas], bins=50, color="#55A868")
    axes[1].set(title="Box size (% of image area)", xlabel="% of image", ylabel="Boxes", yscale="log")
    for a in axes:
        a.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(figs / "box_statistics.png", dpi=150)

    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(list(report["splits"]), [r["images"] for r in report["splits"].values()], color="#8172B2")
    for i, r in enumerate(report["splits"].values()):
        ax.text(i, r["images"], f'{r["images"]} ({r["percent"]}%)', ha="center", va="bottom")
    ax.set(title="Dataset split", ylabel="Images")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(figs / "split_sizes.png", dpi=150)

    random.seed(0)
    picks = random.sample([k for k, v in all_boxes.items() if v], 6)
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for ax, img_path in zip(axes.flat, picks):
        im = Image.open(img_path).convert("RGB")
        d, (W, H) = ImageDraw.Draw(im), im.size
        for c, x, y, w, h in all_boxes[img_path]:
            col = "red" if names[c].startswith("NO-") else "lime"
            d.rectangle([(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H], outline=col, width=3)
            d.text(((x - w / 2) * W + 3, (y - h / 2) * H + 2), names[c], fill=col)
        ax.imshow(im)
        ax.axis("off")
    fig.suptitle("Sample ground-truth annotations (red = violation class)")
    fig.tight_layout()
    fig.savefig(figs / "sample_annotations.png", dpi=120)

    (figs / "cleaning_report.json").write_text(json.dumps(report, indent=2))
    print(json.dumps({"splits": report["splits"], "issues": report["summary"]}, indent=2))
    print(f"Figures and cleaning_report.json saved to {figs}/")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile split_dataset.py
"""Clean the raw dataset and re-split it without leakage.

    python split_dataset.py --src dataset --dst dataset_clean

Why: the Roboflow export has 2,801 images but only ~700 source photos. Each source
appears up to 10 times as augmented copies (same name before ".rf."), and video
frames (names like "construction-4-_mp4-19_jpg") are near-identical. The original
93/4/3 split puts copies of the same photo/video in both train and test, which
inflates test scores.

What it does:
  1. Groups images by source photo, and video frames by source video.
  2. Splits the GROUPS 70/15/15 (seeded), so no source appears in two splits.
  3. Keeps all augmented copies in train (free augmentation), but only one copy
     per source photo in val/test, so every evaluation image is independent.
  4. Drops corrupt images and rewrites labels without malformed/duplicate lines.
"""
import argparse
import random
import re
import shutil
from collections import defaultdict
from pathlib import Path

from PIL import Image

from data_cleaning import IMG_EXT, check_label_file

NAMES = ['Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 'Person',
         'Safety Cone', 'Safety Vest', 'machinery', 'vehicle']
RATIOS = {"train": 0.70, "valid": 0.15, "test": 0.15}


def source_of(name):
    """Original photo name, i.e. the part before Roboflow's '.rf.<hash>'."""
    return name.split(".rf.")[0]


def group_of(name):
    """Video frames share their video as a group; other images group by source photo."""
    m = re.match(r"(.+_mp4)-\d+_jpg$", source_of(name))
    return m.group(1) if m else source_of(name)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--src", default="dataset")
    p.add_argument("--dst", default="dataset_clean")
    p.add_argument("--seed", type=int, default=42)
    args = p.parse_args()
    src, dst = Path(args.src), Path(args.dst)

    groups = defaultdict(list)  # group -> [(image, label)]
    dropped = 0
    for split in ["train", "valid", "test"]:
        for img in sorted((src / split / "images").iterdir()):
            if img.suffix.lower() not in IMG_EXT:
                continue
            try:
                with Image.open(img) as im:
                    im.verify()
            except Exception:
                dropped += 1
                continue
            groups[group_of(img.name)].append((img, src / split / "labels" / f"{img.stem}.txt"))

    keys = sorted(groups)
    random.Random(args.seed).shuffle(keys)
    n_imgs = sum(len(v) for v in groups.values())
    assign, filled = {}, {"train": 0, "valid": 0, "test": 0}
    for k in keys:  # fill test, then valid, then train, by image count
        target = next((s for s in ["test", "valid"] if filled[s] < RATIOS[s] * n_imgs), "train")
        assign[k] = target
        filled[target] += len(groups[k])

    if dst.exists():
        shutil.rmtree(dst)
    stats = {s: {"images": 0, "groups": 0, "fixed_labels": 0} for s in RATIOS}
    for k, items in groups.items():
        split = assign[k]
        if split != "train":  # one copy per source photo in val/test
            seen, kept = set(), []
            for img, lbl in items:
                if source_of(img.name) not in seen:
                    seen.add(source_of(img.name))
                    kept.append((img, lbl))
            items = kept
        stats[split]["groups"] += 1
        for img, lbl in items:
            (dst / split / "images").mkdir(parents=True, exist_ok=True)
            (dst / split / "labels").mkdir(parents=True, exist_ok=True)
            shutil.copy2(img, dst / split / "images" / img.name)
            boxes, problems = check_label_file(lbl, len(NAMES)) if lbl.exists() else ([], [])
            stats[split]["fixed_labels"] += bool(problems)
            (dst / split / "labels" / lbl.name).write_text(
                "".join(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n" for c, x, y, w, h in boxes))
            stats[split]["images"] += 1

    (dst / "data.yaml").write_text(
        "train: train/images\nval: valid/images\ntest: test/images\n\n"
        f"nc: {len(NAMES)}\nnames: {NAMES}\n")
    print(f"{len(groups)} source groups, {dropped} corrupt images dropped")
    for s, v in stats.items():
        print(f"  {s:5s}: {v['images']:5d} images from {v['groups']:4d} groups, {v['fixed_labels']} labels repaired")
    print(f"Wrote {dst}/ with data.yaml")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train.py
"""Train several YOLO models on the same dataset so they can be compared.

    python train.py --data dataset/data.yaml --models yolov8n.pt yolov8s.pt yolov5su.pt
"""
import argparse
from pathlib import Path

from ultralytics import YOLO


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", required=True, help="path to data.yaml (YOLO format)")
    p.add_argument("--models", nargs="+", default=["yolov8n.pt", "yolov8s.pt"])
    p.add_argument("--epochs", type=int, default=50)
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--batch", type=int, default=16)
    args = p.parse_args()

    for weights in args.models:
        name = Path(weights).stem
        print(f"\n=== Training {name} ===")
        YOLO(weights).train(
            data=args.data,
            epochs=args.epochs,
            imgsz=args.imgsz,
            batch=args.batch,
            project=str(Path("runs").resolve()),  # absolute, else Ultralytics nests it under runs/detect/
            name=name,
            exist_ok=True,
            patience=15,
        )


if __name__ == "__main__":
    main()

In [ ]:
%%writefile evaluate.py
"""Compare every trained model in runs/ on the test split.

    python evaluate.py --data dataset_clean/data.yaml

Writes:
  results.csv            precision, recall, mAP, size, FPS, violation-alarm metrics
  results_per_class.csv  mAP@0.5 per class per model
  figures/comparison/    bar chart, accuracy-vs-speed plot, per-class heatmap
"""
import argparse
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import torch
import yaml
from ultralytics import YOLO
from ultralytics.utils.torch_utils import get_flops

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
VIOLATION_PREFIX = "NO-"


def split_images(data_yaml, split):
    cfg = yaml.safe_load(Path(data_yaml).read_text(encoding="utf-8"))
    root = Path(cfg.get("path", Path(data_yaml).parent))
    folder = root / cfg[split]
    return sorted(f for f in folder.rglob("*") if f.suffix.lower() in IMG_EXT)


def measure_fps(model, imgs, imgsz, n=50):
    imgs = imgs[:n]
    model.predict(str(imgs[0]), imgsz=imgsz, verbose=False)  # warm-up
    start = time.perf_counter()
    for im in imgs:
        model.predict(str(im), imgsz=imgsz, verbose=False)
    return len(imgs) / (time.perf_counter() - start)


def violation_alarm(model, imgs, imgsz, conf=0.5):
    """Image-level check: does the model raise an alarm exactly when a NO-* object is present?"""
    viol_ids = {i for i, n in model.names.items() if n.startswith(VIOLATION_PREFIX)}
    tp = fp = fn = tn = 0
    for img in imgs:
        lbl = img.parent.parent / "labels" / f"{img.stem}.txt"
        lines = lbl.read_text().split("\n") if lbl.exists() else []
        truth = any(l.split() and int(l.split()[0]) in viol_ids for l in lines)
        boxes = model.predict(str(img), imgsz=imgsz, conf=conf, verbose=False)[0].boxes
        pred = any(int(c) in viol_ids for c in boxes.cls)
        tp += truth and pred
        fp += pred and not truth
        fn += truth and not pred
        tn += not truth and not pred
    return {"alarm_precision": round(tp / max(tp + fp, 1), 3),
            "alarm_recall": round(tp / max(tp + fn, 1), 3),
            "alarm_accuracy": round((tp + tn) / max(len(imgs), 1), 3)}


def plot(df, per_class, out):
    out.mkdir(parents=True, exist_ok=True)
    models = df["model"]

    fig, ax = plt.subplots(figsize=(8, 4))
    x = range(len(df))
    for i, col in enumerate(["precision", "recall", "mAP@0.5", "mAP@0.5:0.95"]):
        bars = ax.bar([j + (i - 1.5) * 0.2 for j in x], df[col], 0.2, label=col)
        ax.bar_label(bars, fmt="%.2f", fontsize=7, padding=1)
    ax.set_xticks(list(x), models)
    ax.set_ylim(0, 1)
    ax.set_title("Detection metrics on the test split")
    ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.1))
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(out / "metrics_bar.png", dpi=150)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(df["FPS"], df["mAP@0.5"], s=df["params_M"] * 25 + 20, alpha=0.7)
    for _, r in df.iterrows():
        ax.annotate(r["model"], (r["FPS"], r["mAP@0.5"]), xytext=(6, 4), textcoords="offset points", fontsize=9)
    ax.set(xlabel=f"FPS ({df['device'].iloc[0]})", ylabel="mAP@0.5",
           title="Accuracy vs speed (marker size = parameters)")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(out / "accuracy_vs_speed.png", dpi=150)

    fig, ax = plt.subplots(figsize=(1.6 * len(per_class.columns) + 3, 5))
    im = ax.imshow(per_class.values, cmap="viridis", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(per_class.columns)), per_class.columns)
    ax.set_yticks(range(len(per_class.index)), per_class.index)
    for i in range(per_class.shape[0]):
        for j in range(per_class.shape[1]):
            v = per_class.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if v < 0.6 else "black", fontsize=8)
    fig.colorbar(im, ax=ax, label="mAP@0.5")
    ax.set_title("Per-class mAP@0.5")
    fig.tight_layout()
    fig.savefig(out / "per_class_heatmap.png", dpi=150)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", required=True)
    p.add_argument("--imgsz", type=int, default=None, help="default: the size each model was trained at")
    args = p.parse_args()

    cfg = yaml.safe_load(Path(args.data).read_text(encoding="utf-8"))
    split = "test" if cfg.get("test") else "val"
    imgs = split_images(args.data, split)
    device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

    rows, per_class = [], {}
    for best in sorted(Path("runs").glob("*/weights/best.pt")):
        name = best.parent.parent.name
        print(f"Evaluating {name} on '{split}' split ({len(imgs)} images)")
        model = YOLO(str(best))
        imgsz = args.imgsz or model.overrides.get("imgsz", 640)
        m = model.val(data=args.data, split=split, imgsz=imgsz, verbose=False, plots=False)
        params = sum(p.numel() for p in model.model.parameters())
        gflops = get_flops(model.model, imgsz)
        rows.append({
            "model": name,
            "precision": round(m.box.mp, 3),
            "recall": round(m.box.mr, 3),
            "mAP@0.5": round(m.box.map50, 3),
            "mAP@0.5:0.95": round(m.box.map, 3),
            "params_M": round(params / 1e6, 2),
            "GFLOPs": round(gflops, 1),
            "imgsz": imgsz,
            "device": device,
            "FPS": round(measure_fps(model, imgs, imgsz), 1),
            **violation_alarm(model, imgs, imgsz),
        })
        per_class[name] = {model.names[int(c)]: round(m.box.class_result(i)[2], 3)
                           for i, c in enumerate(m.box.ap_class_index)}

    if not rows:
        print("No trained models found in runs/*/weights/best.pt")
        return
    df = pd.DataFrame(rows)
    pc = pd.DataFrame(per_class)
    df.to_csv("results.csv", index=False)
    pc.to_csv("results_per_class.csv")
    plot(df, pc, Path("figures/comparison"))
    print("\n", df.to_string(index=False))
    print("\nPer-class mAP@0.5:\n", pc.to_string())
    print("\nSaved results.csv, results_per_class.csv and figures/comparison/")


if __name__ == "__main__":
    main()

## 3. Dataset: unzip, clean, re-split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/css-data.zip'   # change if you put it elsewhere
!unzip -q -o "$ZIP" -d /content/dataset

In [ ]:
!python split_dataset.py --src dataset --dst dataset_clean --seed 42
!python data_cleaning.py --data dataset_clean/data.yaml --out figures/data_clean

## 4. Train four models
Same data, epochs, image size and seed for all four, so the comparison is fair.
`patience=15` stops a model early if validation mAP stops improving.

In [ ]:
!python train.py --data dataset_clean/data.yaml --models yolov5su.pt yolov8n.pt yolov8s.pt yolo11n.pt --epochs 30 --imgsz 640

## 5. Evaluate and compare

In [ ]:
!python evaluate.py --data dataset_clean/data.yaml
import pandas as pd
display(pd.read_csv('results.csv'))
display(pd.read_csv('results_per_class.csv', index_col=0))

In [ ]:
from IPython.display import Image, display
for f in ['metrics_bar', 'accuracy_vs_speed', 'per_class_heatmap']:
    display(Image(f'figures/comparison/{f}.png', width=700))

## 6. Download
Packs the trained runs (weights, training curves, confusion matrices, PR curves), the
results tables and all figures. A copy also goes to your Drive in case the download fails.

In [ ]:
!zip -q -r results_gpu.zip runs figures results.csv results_per_class.csv -x "runs/*/weights/last.pt"
!cp results_gpu.zip /content/drive/MyDrive/
from google.colab import files
files.download('results_gpu.zip')

## 7. Back on your laptop
1. Unzip `results_gpu.zip` into the `ppe-vision` folder (replace existing files when asked).
2. `python -m streamlit run app.py` and pick any of the four models in the sidebar.
3. Optional: `python evaluate.py --data dataset_clean/data.yaml` to also measure CPU FPS
   (run `python split_dataset.py` first if `dataset_clean/` doesn't exist yet).